# Chapitre 1 · Faire parler une machine

Notebook du chapitre 1 de *Construire un LLM de zéro*. Tu entraînes un mini
modèle de langage au niveau caractère sur trente fables de Jean de La Fontaine,
puis tu le fais écrire.

**Comment travailler.** D'abord la leçon : tout le code du chapitre, complet et
prêt à exécuter, dans l'ordre du livre. Lis, exécute, modifie pour voir. À la
fin, la section **Exercices** : trois défis à trous, du plus simple (●) au plus
costaud (●●●), validés par des `assert`.

Tout tourne **sans GPU et sans connexion internet** : le corpus est embarqué dans
une cellule, et l'entraînement prend environ une minute sur un ordinateur portable.

In [1]:
import time

import torch
import torch.nn as nn
import torch.nn.functional as F


def get_device():
    """Renvoie le meilleur processeur disponible : cuda -> mps -> cpu."""
    if torch.cuda.is_available():
        return torch.device("cuda")      # GPU NVIDIA (Colab, cloud)
    if torch.backends.mps.is_available():
        return torch.device("mps")       # puce Apple Silicon (Mac M1-M4)
    return torch.device("cpu")           # sinon, le processeur classique


device = get_device()
torch.manual_seed(42)                    # même hasard pour tous : résultats reproductibles
print(f"PyTorch {torch.__version__} · calcul sur : {device}")

PyTorch 2.11.0+cu128 · calcul sur : cuda


## 1. Le corpus : trente fables de La Fontaine

Notre matière première est un **texte**, embarqué directement dans la cellule
suivante : trente fables de Jean de La Fontaine (1621-1695), un texte du
**domaine public**. Source : Wikisource, *Fables de La Fontaine*, édition 1874
(Charles Delagrave), typographie légèrement simplifiée (apostrophes droites,
tirets simples, notes retirées).

Comme les vrais labos : eux entraînent sur des téraoctets de web filtré, nous
sur 38 Ko de fables. La différence est l'échelle, pas le principe. On construira
un vrai corpus au chapitre 13.

In [ ]:
corpus = """\
LA CIGALE ET LA FOURMI
La cigale, ayant chanté
Tout l'été,
Se trouva fort dépourvue
Quand la bise fut venue :
Pas un seul petit morceau
De mouche ou de vermisseau.
Elle alla crier famine
Chez la fourmi, sa voisine,
La priant de lui prêter
Quelque grain pour subsister
Jusqu'à la saison nouvelle.
Je vous paierai, lui dit-elle,
Avant l'oût, foi d'animal,
Intérêt et principal.
La fourmi n'est pas prêteuse :
C'est là son moindre défaut.
Que faisiez-vous au temps chaud ?
Dit-elle à cette emprunteuse. -
Nuit et jour à tout venant
Je chantais, ne vous déplaise. -
Vous chantiez, j'en suis fort aise !
Eh bien ! dansez maintenant.


LE CORBEAU ET LE RENARD
Maître corbeau, sur un arbre perché,
Tenait en son bec un fromage.
Maître renard, par l'odeur alléché,
Lui tint à peu près ce langage :
Hé ! bonjour, monsieur du corbeau.
Que vous êtes joli ! que vous me semblez beau !
Sans mentir, si votre ramage
Se rapporte à votre plumage,
Vous êtes le phénix des hôtes de ces bois.
À ces mots le corbeau ne se sent pas de joie ;
Et, pour montrer sa belle voix,
Il ouvre un large bec, laisse tomber sa proie.
Le renard s'en saisit, et dit : Mon bon monsieur,
Apprenez que tout flatteur
Vit aux dépens de celui qui l'écoute :
Cette leçon vaut bien un fromage, sans doute.
Le corbeau, honteux et confus,
Jura, mais un peu tard, qu'on ne l'y prendrait plus.


LA GRENOUILLE QUI SE VEUT FAIRE AUSSI GROSSE QUE LE BŒUF
Une grenouille vit un bœuf
Qui lui sembla de belle taille.
Elle, qui n'était pas grosse en tout comme un œuf,
Envieuse, s'étend, et s'enfle, et se travaille
Pour égaler l'animal en grosseur ;
Disant : Regardez bien, ma sœur ;
Est-ce assez ? dites-moi ; n'y suis-je point encore ? -
Nenni. - M'y voici donc ? - Point du tout. - M'y voilà ? -
Vous n'en approchez point. La chétive pécore
S'enfla si bien qu'elle creva.
Le monde est plein de gens qui ne sont pas plus sages :
Tout bourgeois veut bâtir comme les grands seigneurs,
Tout petit prince a des ambassadeurs,
Tout marquis veut avoir des pages.


LE LOUP ET LE CHIEN
Un loup n'avait que les os et la peau,
Tant les chiens faisaient bonne garde.
Ce loup rencontre un dogue aussi puissant que beau,
Gras, poli, qui s'était fourvoyé par mégarde.
L'attaquer, le mettre en quartiers,
Sire loup l'eût fait volontiers :
Mais il fallait livrer bataille ;
Et le mâtin était de taille
À se défendre hardiment.
Le loup donc l'aborde humblement,
Entre en propos, et lui fait compliment
Sur son embonpoint, qu'il admire.
Il ne tiendra qu'à vous, beau sire,
D'être aussi gras que moi, lui repartit le chien.
Quittez les bois, vous ferez bien :
Vos pareils y sont misérables,
Cancres, hères et pauvres diables,
Dont la condition est de mourir de faim.
Car, quoi ! rien d'assuré ! point de franche lippée !
Tout à la pointe de l'épée !
Suivez-moi, vous aurez un bien meilleur destin.
Le loup reprit : Que me faudra-t-il faire ?
Presque rien, dit le chien : donner la chasse aux gens
Portants bâtons, et mendiants ;
Flatter ceux du logis, à son maître complaire ;
Moyennant quoi votre salaire
Sera force reliefs de toutes les façons,
Os de poulets, os de pigeons ;
Sans parler de mainte caresse.
Le loup déjà se forge une félicité
Qui le fait pleurer de tendresse.
Chemin faisant il vit le cou du chien pelé.
Qu'est-ce là ? lui dit-il. - Rien. - Quoi ! rien ! - Peu de chose. -
Mais encor ? - Le collier dont je suis attaché
De ce que vous voyez est peut-être la cause.
Attaché ! dit le loup : vous ne courez donc pas
Où vous voulez ? - Pas toujours ; mais qu'importe ?
Il importe si bien, que de tous vos repas
Je ne veux en aucune sorte,
Et ne voudrais pas même à ce prix d'un trésor.
Cela dit, maître loup s'enfuit, et court encor.


LA BESACE
Jupiter dit un jour : Que tout ce qui respire
S'en vienne comparaître aux pieds de ma grandeur :
Si dans son composé quelqu'un trouve à redire,
Il peut le déclarer sans peur ;
Je mettrai remède à la chose.
Venez, singe ; parlez le premier, et pour cause :
Voyez ces animaux, faites comparaison
De leurs beautés avec les vôtres.
Êtes-vous satisfait ? - Moi, dit-il ; pourquoi non ?
N'ai-je pas quatre pieds aussi bien que les autres ?
Mon portrait jusqu'ici ne m'a rien reproché :
Mais pour mon frère l'ours, on ne l'a qu'ébauché ;
Jamais, s'il me veut croire, il ne se fera peindre.
L'ours venant là-dessus, on crut qu'il s'allait plaindre.
Tant s'en faut : de sa forme il se loua très-fort ;
Glosa sur l'éléphant, dit qu'on pourrait encor
Ajouter à sa queue, ôter à ses oreilles ;
Que c'était une masse informe et sans beauté.
L'éléphant étant écouté,
Tout sage qu'il était, dit des choses pareilles :
Il jugea qu'à son appétit
Dame baleine était trop grosse.
Dame fourmi trouva le ciron trop petit,
Se croyant, pour elle, un colosse.
Jupin les renvoya s'étant censurés tous,
Du reste, contents d'eux. Mais, parmi les plus fous,
Notre espèce excella ; car, tout ce que nous sommes,
Lynx envers nos pareils, et taupes envers nous,
Nous nous pardonnons tout, et rien aux autres hommes :
On se voit d'un autre œil qu'on ne voit son prochain.
Le fabricateur souverain
Nous créa besaciers tous de même manière,
Tant ceux du temps passé que du temps d'aujourd'hui :
Il fit pour nos défauts la poche de derrière,
Et celle de devant pour les défauts d'autrui.


LE LOUP ET L'AGNEAU
La raison du plus fort est toujours la meilleure :
Nous l'allons montrer tout à l'heure.
Un agneau se désaltérait
Dans le courant d'une onde pure.
Un loup survint à jeun, qui cherchait aventure,
Et que la faim en ces lieux attirait.
Qui te rend si hardi de troubler mon breuvage ?
Dit cet animal plein de rage :
Tu seras châtié de ta témérité.
Sire, répond l'agneau, que Votre Majesté
Ne se mette pas en colère ;
Mais plutôt qu'elle considère
Que je me vas désaltérant
Dans le courant,
Plus de vingt pas au-dessous d'elle ;
Et que, par conséquent, en aucune façon
Je ne puis troubler sa boisson.
Tu la troubles ! reprit cette bête cruelle ;
Et je sais que de moi tu médis l'an passé.
Comment l'aurais-je fait, si je n'étais pas né ?
Reprit l'agneau : je tette encore ma mère. -
Si ce n'est toi, c'est donc ton frère. -
Je n'en ai point. - C'est donc quelqu'un des tiens ;
Car vous ne m'épargnez guère,
Vous, vos bergers et vos chiens.
On me l'a dit : il faut que je me venge.
Là-dessus, au fond des forêts
Le loup l'emporte, et puis le mange,
Sans autre forme de procès.


LA MORT ET LE BÛCHERON
Un pauvre bucheron, tout couvert de ramée,
Sous le faix du fagot aussi bien que des ans,
Gémissant et courbé, marchait à pas pesants,
Et tâchait de gagner sa chaumine enfumée.
Enfin, n'en pouvant plus d'effort et de douleur,
Il met bas son fagot, il songe à son malheur.
Quel plaisir a-t-il eu depuis qu'il est au monde ?
En est-il un plus pauvre en la machine ronde ?
Point de pain quelquefois, et jamais de repos :
Sa femme, ses enfants, les soldats, les impôts,
Le créancier, et la corvée,
Lui font d'un malheureux la peinture achevée.
Il appelle la Mort. Elle vient sans tarder,
Lui demande ce qu'il faut faire.
C'est, dit-il, afin de m'aider
À recharger ce bois ; tu ne tarderas guère.
Le trépas vient tout guérir ;
Mais ne bougeons d'où nous sommes :
Plutôt souffrir que mourir,
C'est la devise des hommes.


LE RENARD ET LA CIGOGNE
Compère le renard se mit un jour en frais,
Et retint à dîner commère la cigogne.
Le régal fut petit et sans beaucoup d'apprêts :
Le galant, pour toute besogne,
Avait un brouet clair ; il vivait chichement.
Ce brouet fut par lui servi sur une assiette :
La cigogne au long bec n'en put attraper miette ;
Et le drôle eut lapé le tout en un moment.
Pour se venger de cette tromperie,
À quelque temps de là la cigogne le prie.
Volontiers, lui dit-il ; car avec mes amis
Je ne fais point cérémonie.
À l'heure dite, il courut au logis
De la cigogne son hôtesse ;
Loua très-fort sa politesse ;
Trouva le dîner cuit à point :
Bon appétit surtout ; renards n'en manquent point.
Il se réjouissait à l'odeur de la viande
Mise en menus morceaux, et qu'il croyait friande.
On servit, pour l'embarrasser,
En un vase à long col et d'étroite embouchure.
Le bec de la cigogne y pouvait bien passer ;
Mais le museau du sire était d'autre mesure.
Il lui fallut à jeun retourner au logis,
Honteux comme un renard qu'une poule aurait pris,
Serrant la queue, et portant bas l'oreille.
Trompeurs, c'est pour vous que j'écris :
Attendez-vous à la pareille.


LE CHÊNE ET LE ROSEAU
Le chêne un jour dit au roseau :
Vous avez bien sujet d'accuser la nature ;
Un roitelet pour vous est un pesant fardeau :
Le moindre vent qui d'aventure
Fait rider la face de l'eau,
Vous oblige à baisser la tête ;
Cependant que mon front, au Caucase pareil,
Non content d'arrêter les rayons du soleil,
Brave l'effort de la tempête.
Tout vous est aquilon, tout me semble zéphyr.
Encor si vous naissiez à l'abri du feuillage
Dont je couvre le voisinage,
Vous n'auriez pas tant à souffrir,
Je vous défendrais de l'orage :
Mais vous naissez le plus souvent
Sur les humides bords des royaumes du vent.
La nature envers vous me semble bien injuste.
Votre compassion, lui répondit l'arbuste,
Part d'un bon naturel ; mais quittez ce souci :
Les vents me sont moins qu'à vous redoutables ;
Je plie et ne romps pas. Vous avez jusqu'ici
Contre leurs coups épouvantables
Résisté sans courber le dos ;
Mais attendons la fin. Comme il disait ces mots,
Du bout de l'horizon accourt avec furie
Le plus terrible des enfants
Que le Nord eût portés jusque-là dans ses flancs.
L'arbre tient bon ; le roseau plie.
Le vent redouble ses efforts,
Et fait si bien qu'il déracine
Celui de qui la tête au ciel était voisine,
Et dont les pieds touchaient à l'empire des morts.


LE LION ET LE RAT
Il faut, autant qu'on peut, obliger tout le monde :
On a souvent besoin d'un plus petit que soi.
De cette vérité deux fables feront foi ;
Tant la chose en preuves abonde.
Entre les pattes d'un lion
Un rat sortit de terre assez à l'étourdie.
Le roi des animaux, en cette occasion,
Montra ce qu'il était, et lui donna la vie.
Ce bienfait ne fut pas perdu.
Quelqu'un aurait-il jamais cru
Qu'un lion d'un rat eût affaire ?
Cependant il advint qu'au sortir des forêts
Ce lion fut pris dans des rets,
Dont ses rugissements ne le purent défaire.
Sire rat accourut, et fit tant par ses dents
Qu'une maille rongée emporta tout l'ouvrage.
Patience et longueur de temps
Font plus que force ni que rage.


LA COLOMBE ET LA FOURMI
L'autre exemple est tiré d'animaux plus petits.
Le long d'un clair ruisseau buvait une colombe,
Quand sur l'eau se penchant une fourmis y tombe ;
Et dans cet océan on eût vu la fourmis
S'efforcer, mais en vain, de regagner la rive.
La colombe aussitôt usa de charité :
Un brin d'herbe dans l'eau par elle étant jeté,
Ce fut un promontoire où la fourmis arrive.
Elle se sauve. Et là-dessus
Passe un certain croquant qui marchait les pieds nus :
Ce croquant, par hasard, avait une arbalète.
Dès qu'il voit l'oiseau de Vénus,
Il le croit en son pot, et déjà lui fait fête.
Tandis qu'à le tuer mon villageois s'apprête,
La fourmi le pique au talon.
Le vilain retourne la tête :
La colombe l'entend, part, et tire de long.
Le souper du croquant avec elle s'envole :
Point de pigeon pour une obole.


LE LIÈVRE ET LA TORTUE
Rien ne sert de courir ; il faut partir à point :
Le lièvre et la tortue en sont un témoignage.
Gageons, dit celle-ci, que vous n'atteindrez point
Sitôt que moi ce but. Sitôt ! êtes-vous sage ?
Repartit l'animal léger :
Ma commère, il vous faut purger
Avec quatre grains d'ellébore.
- Sage ou non, je parie encore.
Ainsi fut fait ; et de tous deux
On mit près du but les enjeux.
Savoir quoi, ce n'est pas l'affaire,
Ni de quel juge l'on convint.
Notre lièvre n'avait que quatre pas à faire ;
J'entends de ceux qu'il fait lorsque, près d'être atteint,
Il s'éloigne des chiens, les renvoie aux calendes,
Et leur fait arpenter les landes.
Ayant, dis-je, du temps de reste pour brouter,
Pour dormir, et pour écouter
D'où vient le vent, il laisse la tortue
Aller son train de sénateur.
Elle part, elle s'évertue ;
Elle se hâte avec lenteur.
Lui cependant méprise une telle victoire,
Tient la gageure à peu de gloire,
Croit qu'il y va de son honneur
De partir tard. Il broute, il se repose ;
Il s'amuse à toute autre chose
Qu'à la gageure. À la fin, quand il vit
Que l'autre touchait presque au bout de la carrière,
Il partit comme un trait ; mais les élans qu'il fit
Furent vains : la tortue arriva la première.
Eh bien ! lui cria-t-elle, avais-je pas raison ?
De quoi vous sert votre vitesse ?
Moi l'emporter ! et que serait-ce
Si vous portiez une maison ?


LE RENARD ET LES RAISINS
Certain renard gascon, d'autres disent normand,
Mourant presque de faim, vit au haut d'une treille
Des raisins, mûrs apparemment,
Et couverts d'une peau vermeille.
Le galant en eût fait volontiers un repas ;
Mais comme il n'y pouvait atteindre :
Ils sont trop verts, dit-il, et bons pour des goujats.
Fit-il pas mieux que de se plaindre ?


LE HÉRON
Un jour, sur ses longs pieds, allait je ne sais où
Le héron au long bec emmanché d'un long cou :
Il côtoyait une rivière.
L'onde était transparente ainsi qu'aux plus beaux jours ;
Ma commère la carpe y faisait mille tours
Avec le brochet son compère.
Le héron en eût fait aisément son profit :
Tous approchaient du bord ; l'oiseau n'avait qu'à prendre.
Mais il crut mieux faire d'attendre
Qu'il eût un peu plus d'appétit :
Il vivait de régime, et mangeait à ses heures.
Après quelques moments l'appétit vint : l'oiseau,
S'approchant du bord, vit sur l'eau
Des tanches qui sortaient du fond de ces demeures.
Le mets ne lui plut pas, il s'attendait à mieux,
Et montrait un goût dédaigneux
Comme le rat du bon Horace.
Moi, des tanches ! dit-il ; moi, héron, que je fasse
Une si pauvre chère ! Et pour qui me prend-on ?
La tanche rebutée, il trouva du goujon.
Du goujon ! c'est bien là le dîner d'un héron !
J'ouvrirais pour si peu le bec ! aux dieux ne plaise !
Il l'ouvrit pour bien moins : tout alla de façon
Qu'il ne vit plus aucun poisson.
La faim le prit : il fut tout heureux et tout aise
De rencontrer un limaçon.
Ne soyons pas si difficiles :
Les plus accommodants, ce sont les plus habiles ;
On hasarde de perdre en voulant trop gagner.
Gardez-vous de rien dédaigner,
Surtout quand vous avez à peu près votre compte.
Bien des gens y sont pris. Ce n'est pas aux hérons
Que je parle : écoutez, humains, un autre conte :
Vous verrez que chez vous j'ai puisé ces leçons.


LA LAITIÈRE ET LE POT AU LAIT
Perrette, sur sa tête ayant un pot au lait
Bien posé sur un coussinet,
Prétendait arriver sans encombre à la ville.
Légère et court vêtue, elle allait à grands pas,
Ayant mis ce jour-là, pour être plus agile,
Cotillon simple et souliers plats.
Notre laitière ainsi troussée
Comptait déjà dans sa pensée
Tout le prix de son lait ; en employait l'argent ;
Achetait un cent d'œufs ; faisait triple couvée :
La chose allait à bien par son soin diligent.
Il m'est, disait-elle, facile
D'élever des poulets autour de ma maison ;
Le renard sera bien habile
S'il ne m'en laisse assez pour avoir un cochon.
Le porc à s'engraisser coûtera peu de son ;
Il était, quand je l'eus, de grosseur raisonnable :
J'aurai, le revendant, de l'argent bel et bon.
Et qui m'empêchera de mettre en notre étable,
Vu le prix dont il est, une vache et son veau,
Que je verrai sauter au milieu du troupeau ?
Perrette là-dessus saute aussi, transportée :
Le lait tombe ; adieu veau, vache, cochon, couvée.
La dame de ces biens, quittant d'un œil marri
Sa fortune ainsi répandue,
Va s'excuser à son mari,
En grand danger d'être battue.
Le récit en farce en fut fait ;
On l'appela le Pot au lait.
Quel esprit ne bat la campagne ?
Qui ne fait châteaux en Espagne ?
Picrochole, Pyrrhus, la laitière, enfin tous,
Autant les sages que les fous.
Chacun songe en veillant ; il n'est rien de plus doux
Une flatteuse erreur emporte alors nos âmes ;
Tout le bien du monde est à nous,
Tous les honneurs, toutes les femmes.
Quand je suis seul, je fais au plus brave un défi ;
Je m'écarte, je vais détrôner le sophi ;
On m'élit roi, mon peuple m'aime ;
Les diadèmes vont sur ma tête pleuvant :
Quelque accident fait-il que je rentre en moi-même ;
Je suis Gros-Jean comme devant.


LE COCHE ET LA MOUCHE
Dans un chemin montant, sablonneux, malaisé,
Et de tous les côtés au soleil exposé,
Six forts chevaux tiraient un coche.
Femmes, moine, vieillards, tout était descendu :
L'attelage suait, soufflait, était rendu.
Une mouche survient, et des chevaux s'approche,
Prétend les animer par son bourdonnement,
Pique l'un, pique l'autre, et pense à tout moment
Qu'elle fait aller la machine,
S'assied sur le timon, sur le nez du cocher.
Aussitôt que le char chemine,
Et qu'elle voit les gens marcher,
Elle s'en attribue uniquement la gloire,
Va, vient, fait l'empressée : il semble que ce soit
Un sergent de bataille allant en chaque endroit
Faire avancer ses gens et hâter la victoire.
La mouche, en ce commun besoin,
Se plaint qu'elle agit seule, et qu'elle a tout le soin ;
Qu'aucun n'aide aux chevaux à se tirer d'affaire.
Le moine disait son bréviaire :
Il prenait bien son temps ! une femme chantait :
C'était bien de chansons qu'alors il s'agissait !
Dame mouche s'en va chanter à leurs oreilles,
Et fait cent sottises pareilles.
Après bien du travail, le coche arrive au haut.
Respirons maintenant ! dit la mouche aussitôt :
J'ai tant fait que nos gens sont enfin dans la plaine.
Çà, messieurs les chevaux, payez-moi de ma peine.
Ainsi certaines gens, faisant les empressés,
S'introduisent dans les affaires :
Ils font partout les nécessaires,
Et, partout importuns, devraient être chassés.


LE SAVETIER ET LE FINANCIER
Un savetier chantait du matin jusqu'au soir :
C'était merveille de le voir,
Merveille de l'ouïr ; il faisait des passages :
Plus content qu'aucun des sept sages.
Son voisin, au contraire, étant tout cousu d'or,
Chantait peu, dormait moins encor :
C'était un homme de finance.
Si sur le point du jour parfois il sommeillait,
Le savetier alors en chantant l'éveillait ;
Et le financier se plaignait
Que les soins de la Providence
N'eussent pas au marché fait vendre le dormir,
Comme le manger et le boire.
En son hôtel il fait venir
Le chanteur, et lui dit : Or çà, sire Grégoire,
Que gagnez-vous par an ? Par an ! ma foi, monsieur
Dit avec un ton de rieur
Le gaillard savetier, ce n'est point ma manière
De compter de la sorte ; et je n'entasse guère
Un jour sur l'autre : il suffit qu'à la fin
J'attrape le bout de l'année ;
Chaque jour amène son pain. -
Eh bien ! que gagnez-vous, dites-moi, par journée ?
Tantôt plus, tantôt moins : le mal est que toujours
(Et sans cela nos gains seraient assez honnêtes),
Le mal est que dans l'an s'entremêlent des jours
Qu'il faut chômer ; on nous ruine en fêtes :
L'une fait tort à l'autre ; et monsieur le curé
De quelque nouveau saint charge toujours son prône.
Le financier, riant de sa naïveté,
Lui dit : Je vous veux mettre aujourd'hui sur le trône.
Prenez ces cent écus ; gardez-les avec soin,
Pour vous en servir au besoin.
Le savetier crut voir tout l'argent que la terre
Avait depuis plus de cent ans,
Produit pour l'usage des gens.
Il retourne chez lui : dans sa cave il enserre
L'argent, et sa joie à la fois.
Plus de chant : il perdit la voix
Du moment qu'il gagna ce qui cause nos peines.
Le sommeil quitta son logis :
Il eut pour hôtes les soucis,
Les soupçons, les alarmes vaines.
Tout le jour il avait l'œil au guet ; et la nuit,
Si quelque chat faisait du bruit,
Le chat prenait l'argent. À la fin le pauvre homme
S'en courut chez celui qu'il ne réveillait plus :
Rendez-moi, lui dit-il, mes chansons et mon somme ;
Et reprenez vos cent écus.


LES ANIMAUX MALADES DE LA PESTE
Un mal qui répand la terreur,
Mal que le ciel en sa fureur
Inventa pour punir les crimes de la terre,
La peste (puisqu'il faut l'appeler par son nom),
Capable d'enrichir en un jour l'Achéron,
Faisait aux animaux la guerre.
Ils ne mouraient pas tous, mais tous étaient frappés :
On n'en voyait point d'occupés
À chercher le soutien d'une mourante vie ;
Nul mets n'excitait leur envie ;
Ni loups ni renards n'épiaient
La douce et l'innocente proie ;
Les tourterelles se fuyaient :
Plus d'amour, partant plus de joie.
Le lion tint conseil, et dit : Mes chers amis,
Je crois que le ciel a permis
Pour nos péchés cette infortune.
Que le plus coupable de nous
Se sacrifie aux traits du céleste courroux ;
Peut-être il obtiendra la guérison commune.
L'histoire nous apprend qu'en de tels accidents
On fait de pareils dévouements.
Ne nous flattons donc point ; voyons sans indulgence
L'état de notre conscience.
Pour moi, satisfaisant mes appétits gloutons,
J'ai dévoré force moutons.
Que m'avaient-ils fait ? nulle offense ;
Même il m'est arrivé quelquefois de manger
Le berger.
Je me dévouerai donc, s'il le faut : mais je pense
Qu'il est bon que chacun s'accuse ainsi que moi ;
Car on doit souhaiter, selon toute justice,
Que le plus coupable périsse.
Sire, dit le renard, vous êtes trop bon roi ;
Vos scrupules font voir trop de délicatesse.
Eh bien ! manger moutons, canaille, sotte espèce,
Est-ce un péché ? Non, non. Vous leur fîtes, seigneur,
En les croquant, beaucoup d'honneur ;
Et quant au berger, l'on peut dire
Qu'il était digne de tous maux,
Étant de ces gens-là qui sur les animaux
Se font un chimérique empire.
Ainsi dit le renard ; et flatteurs d'applaudir.
On n'osa trop approfondir
Du tigre, ni de l'ours, ni des autres puissances,
Les moins pardonnables offenses :
Tous les gens querelleurs, jusqu'aux simples mâtins,
Au dire de chacun, étaient de petits saints.
L'âne vint à son tour, et dit : J'ai souvenance
Qu'en un pré de moines passant,
La faim, l'occasion, l'herbe tendre, et, je pense,
Quelque diable aussi me poussant,
Je tondis de ce pré la largeur de ma langue ;
Je n'en avais nul droit, puisqu'il faut parler net.
À ces mots, on cria haro sur le baudet.
Un loup, quelque peu clerc, prouva par sa harangue
Qu'il fallait dévouer ce maudit animal,
Ce pelé, ce galeux, d'où venait tout leur mal.
Sa peccadille fut jugée un cas pendable.
Manger l'herbe d'autrui ! quel crime abominable !
Rien que la mort n'était capable
D'expier son forfait. On le lui fit bien voir.
Selon que vous serez puissant ou misérable,
Les jugements de cour vous rendront blanc ou noir.


LA POULE AUX ŒUFS D'OR
L'avarice perd tout en voulant tout gagner.
Je ne veux, pour le témoigner,
Que celui dont la poule, à ce que dit la fable,
Pondait tous les jours un œuf d'or.
Il crut que, dans son corps, elle avait un trésor ;
Il la tua, l'ouvrit, et la trouva semblable
À celle dont les œufs ne lui rapportaient rien,
S'étant lui-même ôté le plus beau de son bien.
Belle leçon pour les gens chiches !
Pendant ces derniers temps combien en a-t-on vus
Qui du soir au matin sont pauvres devenus
Pour vouloir trop tôt être riches !


L'OURS ET LES DEUX COMPAGNONS
Deux compagnons, pressés d'argent,
À leur voisin fourreur vendirent
La peau d'un ours encor vivant,
Mais qu'ils tueraient bientôt, du moins à ce qu'ils dirent,
C'était le roi des ours au compte de ces gens.
Le marchand à sa peau devait faire fortune ;
Elle garantirait des froids les plus cuisants ;
On en pourrait fourrer plutôt deux robes qu'une.
Dindenaut prisait moins ses moutons qu'eux leur ours :
Leur, à leur compte, et non à celui de la bête.
S'offrant de la livrer au plus tard dans deux jours,
Ils conviennent de prix, et se mettent en quête,
Trouvent l'ours qui s'avance et vient vers eux au trot,
Voilà mes gens frappés comme d'un coup de foudre.
Le marché ne tint pas ; il fallut le résoudre :
D'intérêts contre l'ours, on n'en dit pas un mot.
L'un des deux compagnons grimpe au faîte d'un arbre ;
L'autre, plus froid que n'est un marbre,
Se couche sur le nez, fait le mort, tient son vent,
Ayant quelque part ouï dire
Que l'ours s'acharne peu souvent
Sur un corps qui ne vit, ne meut, ni ne respire.
Seigneur ours, comme un sot, donna dans ce panneau :
Il voit ce corps gisant, le croit privé de vie ;
Et, de peur de supercherie,
Le tourne, le retourne, approche son museau,
Flaire aux passages de l'haleine.
C'est, dit-il, un cadavre ; ôtons-nous, car il sent.
À ces mots, l'ours s'en va dans la forêt prochaine.
L'un de nos deux marchands de son arbre descend,
Court à son compagnon, lui dit que c'est merveille
Qu'il n'ait eu seulement que la peur pour tout mal.
Eh bien ! ajouta-t-il, la peau de l'animal ?
Mais que t'a-t-il dit à l'oreille ?
Car il t'approchait de bien près,
Te retournant avec sa serre.
Il m'a dit qu'il ne faut jamais
Vendre la peau de l'ours qu'on ne l'ait mis par terre.


LE RENARD ET LE BOUC
Capitaine renard allait de compagnie
Avec son ami bouc des plus haut encornés :
Celui-ci ne voyait pas plus loin que son nez ;
L'autre était passé maître en fait de tromperie.
La soif les obligea de descendre en un puits ;
Là chacun d'eux se désaltère.
Après qu'abondamment tous deux en eurent pris,
Le renard dit au bouc : Que ferons-nous, compère ?
Ce n'est pas tout de boire, il faut sortir d'ici.
Lève tes pieds en haut, et tes cornes aussi ;
Mets-les contre le mur : le long de ton échine
Je grimperai premièrement ;
Puis sur tes cornes m'élevant,
À l'aide de cette machine,
De ce lieu-ci je sortirai,
Après quoi je t'en tirerai.
Par ma barbe, dit l'autre, il est bon ; et je loue
Les gens bien sensés comme toi.
Je n'aurais jamais, quant à moi,
Trouvé ce secret, je l'avoue.
Le renard sort du puits, laisse son compagnon,
Et vous lui fait un beau sermon
Pour l'exhorter à patience.
Si le ciel t'eût, dit-il, donné par excellence
Autant de jugement que de barbe au menton,
Tu n'aurais pas, à la légère,
Descendu dans ce puits. Or, adieu ; j'en suis hors :
Tâche de t'en tirer et fais tous les efforts ;
Car, pour moi, j'ai certaine affaire
Qui ne me permet pas d'arrêter en chemin.
En toute chose il faut considérer la fin.


LE CERF SE VOYANT DANS L'EAU
Dans le cristal d'une fontaine
Un cerf se mirant autrefois
Louait la beauté de son bois,
Et ne pouvait qu'avecque peine
Souffrir ses jambes de fuseaux,
Dont il voyait l'objet se perdre dans les eaux.
Quelle proportion de mes pieds à ma tête !
Disait-il en voyant leur ombre avec douleur :
Des taillis les plus hauts mon front atteint le faîte ;
Mes pieds ne me font point d'honneur.
Tout en parlant de la sorte,
Un limier le fait partir.
Il tâche à se garantir ;
Dans les forêts il s'emporte :
Son bois, dommageable ornement,
L'arrêtant à chaque moment,
Nuit à l'office que lui rendent
Ses pieds de qui ses jours dépendent.
Il se dédit alors, et maudit les présents
Que le ciel lui fait tous les ans.
Nous faisons cas du beau, nous méprisons l'utile ;
Et le beau souvent nous détruit.
Ce cerf blâme ses pieds qui le rendent agile ;
Il estime un bois qui lui nuit.


LE LOUP DEVENU BERGER
Un loup qui commençait d'avoir petite part
Aux brebis de son voisinage,
Crut qu'il fallait s'aider de la peau du renard,
Et faire un nouveau personnage
Il s'habille en berger, endosse un hoqueton,
Fait sa houlette d'un bâton,
Sans oublier la cornemuse.
Pour pousser jusqu'au bout la ruse,
Il aurait volontiers écrit sur son chapeau :
"C'est moi qui suis Guillot, berger de ce troupeau."
Sa personne étant ainsi faite,
Et ses pieds de devant posés sur sa houlette,
Guillot le sycophante approche doucement.
Guillot, le vrai Guillot, étendu sur l'herbette,
Dormait alors profondément ;
Son chien dormait aussi, comme aussi sa musette :
La plupart des brebis dormaient pareillement.
L'hypocrite les laissa faire ;
Et, pour pouvoir mener vers son fort les brebis,
Il voulut ajouter la parole aux habits,
Chose qu'il croyait nécessaire ;
Mais cela gâta son affaire :
Il ne put du pasteur contrefaire la voix.
Le ton dont il parla fit retentir les bois,
Et découvrit tout le mystère.
Chacun se réveille à ce son,
Les brebis, le chien, le garçon.
Le pauvre loup, dans cet esclandre,
Empêché par son hoqueton,
Ne put ni fuir ni se défendre.
Toujours par quelque endroit fourbes se laissent prendre
Quiconque est loup agisse en loup ;
C'est le plus certain de beaucoup.


LE RAT DE VILLE, ET LE RAT DES CHAMPS
Autrefois le rat de ville
Invita le rat des champs,
D'une façon fort civile,
À des reliefs d'ortolans.
Sur un tapis de Turquie
Le couvert se trouva mis.
Je laisse à penser la vie
Que firent ces deux amis.
Le régal fut fort honnête ;
Rien ne manquait au festin :
Mais quelqu'un troubla la fête
Pendant qu'ils étaient en train.
À la porte de la salle
Ils entendirent du bruit :
Le rat de ville détale ;
Son camarade le suit.
Le bruit cesse, on se retire :
Rats en campagne aussitôt ;
Et le citadin de dire :
Achevons tout notre rôt.
C'est assez, dit le rustique :
Demain vous viendrez chez moi.
Ce n'est pas que je me pique
De tous vos festins de roi :
Mais rien ne vient m'interrompre ;
Je mange tout à loisir.
Adieu donc. Fi du plaisir
Que la crainte peut corrompre !


LE PETIT POISSON ET LE PÊCHEUR
Petit poisson deviendra grand,
Pourvu que Dieu lui prête vie ;
Mais le lâcher en attendant,
Je tiens pour moi que c'est folie,
Car de le rattraper il n'est pas trop certain.
Un carpeau qui n'était encore que fretin,
Fut pris par un pêcheur au bord d'une rivière.
Tout fait nombre, dit l'homme, en voyant son butin ;
Voilà commencement de chère et de festin :
Mettons-le en notre gibecière.
Le pauvre carpillon lui dit en sa manière :
Que ferez-vous de moi ? je ne saurais fournir
Au plus qu'une demi-bouchée.
Laissez-moi carpe devenir :
Je serai par vous repêchée ;
Quelque gros partisan m'achètera bien cher :
Au lieu qu'il vous en faut chercher
Peut-être encor cent de ma taille
Pour faire un plat : quel plat ! croyez-moi, rien qui vaille.
Rien qui vaille ! eh bien ! soit, repartit le pêcheur ;
Poisson, mon bel ami, qui faites le prêcheur,
Vous irez dans la poêle ; et, vous avez beau dire,
Dès ce soir on vous fera frire.
Un Tiens, vaut, ce dit-on, mieux que deux Tu l'auras :
L'un est sûr, l'autre ne l'est pas.


LE POT DE TERRE ET LE POT DE FER
Le pot de fer proposa
Au pot de terre un voyage.
Celui-ci s'en excusa,
Disant qu'il ferait que sage
De garder le coin du feu :
Car il lui fallait si peu,
Si peu que la moindre chose
De son débris serait cause :
Il n'en reviendrait morceau.
Pour vous, dit-il, dont la peau
Est plus dure que la mienne,
Je ne vois rien qui vous tienne.
Nous vous mettrons à couvert,
Repartit le pot de fer :
Si quelque matière dure
Vous menace d'aventure,
Entre deux je passerai,
Et du coup vous sauverai.
Cette offre le persuade.
Pot de fer son camarade
Se met droit à ses côtés.
Mes gens s'en vont à trois pieds
Clopin clopant comme ils peuvent,
L'un contre l'autre jetés
Au moindre hoquet qu'ils treuvent.
Le pot de terre en souffre ; il n'eut pas fait cent pas
Que par son compagnon il fut mis en éclats,
Sans qu'il eût lieu de se plaindre.
Ne nous associons qu'avecque nos égaux ;
Ou bien il nous faudra craindre
Le destin d'un de ces pots.


LE LABOUREUR ET SES ENFANTS
Travaillez, prenez de la peine :
C'est le fonds qui manque le moins.
Un riche laboureur, sentant sa mort prochaine,
Fit venir ses enfants, leur parla sans témoins.
Gardez-vous, leur dit-il, de vendre l'héritage
Que nous ont laissé nos parents :
Un trésor est caché dedans.
Je ne sais pas l'endroit ; mais un peu de courage
Vous le fera trouver : vous en viendrez à bout.
Remuez votre champ dès qu'on aura fait l'oût :
Creusez, fouillez, bêchez ; ne laissez nulle place
Où la main ne passe et repasse.
Le père mort, les fils vous retournent le champ,
De çà, de là, partout ; si bien qu'au bout de l'an
Il en rapporta davantage.
D'argent, point de caché. Mais le père fut sage
De leur montrer, avant sa mort,
Que le travail est un trésor.


LE MEUNIER, SON FILS, ET L'ÂNE
L'invention des arts étant un droit d'aînesse,
Nous devons l'apologue à l'ancienne Grèce :
Mais ce champ ne se peut tellement moissonner
Que les derniers venus n'y trouvent à glaner.
La feinte est un pays plein de terres désertes ;
Tous les jours nos auteurs y font des découvertes.
Je t'en veux dire un trait assez bien inventé :
Autrefois à Racan Malherbe l'a conté.
Ces deux rivaux d'Horace, héritiers de sa lyre,
Disciples d'Apollon, nos maîtres, pour mieux dire,
Se rencontrant un jour tout seuls et sans témoins
(Comme ils se confiaient leurs pensers et leurs soins),
Racan commence ainsi : Dites-moi, je vous prie,
Vous qui devez savoir les choses de la vie,
Qui par tous ses degrés avez déjà passé,
Et que rien ne doit fuir en cet âge avancé,
À quoi me résoudrai-je ? Il est temps que j'y pense.
Vous connaissez mon bien, mon talent, ma naissance :
Dois-je dans la province établir mon séjour,
Prendre emploi dans l'armée, ou bien charge à la cour ?
Tout au monde est mêlé d'amertume et de charmes :
La guerre a ses douceurs, l'hymen a ses alarmes.
Si je suivais mon goût, je saurais où buter ;
Mais j'ai les miens, la cour, le peuple à contenter.
Malherbe là-dessus : Contenter tout le monde !
Écoutez ce récit avant que je réponde.
J'ai lu dans quelque endroit qu'un meunier et son fils,
L'un vieillard, l'autre enfant, non pas des plus petits,
Mais garçon de quinze ans, si j'ai bonne mémoire,
Allaient vendre leur âne, un certain jour de foire.
Afin qu'il fût plus frais et de meilleur débit,
On lui lia les pieds, on vous le suspendit ;
Puis cet homme et son fils le portent comme un lustre.
Pauvres gens ! idiots ! couple ignorant et rustre !
Le premier qui les vit de rire s'éclata :
Quelle farce, dit-il, vont jouer ces gens-là ?
Le plus âne des trois n'est pas celui qu'on pense.
Le meunier, à ces mots, connaît son ignorance ;
Il met sur pieds sa bête, et la fait détaler.
L'âne, qui goûtait fort l'autre façon d'aller,
Se plaint en son patois. Le meunier n'en a cure,
Il fait monter son fils, il suit : et, d'aventure,
Passent trois bons marchands. Cet objet leur déplut.
Le plus vieux au garçon s'écria tant qu'il put :
Oh là ! oh ! descendez, que l'on ne vous le dise,
Jeune homme, qui menez laquais à barbe grise !
C'était à vous de suivre, au vieillard de monter.
Messieurs, dit le meunier, il vous faut contenter.
L'enfant met pied à terre, et puis le vieillard monte ;
Quand trois filles passant, l'une dit : C'est grand'honte
Qu'il faille voir ainsi clocher ce jeune fils,
Tandis que ce nigaud, comme un évêque assis,
Fait le veau sur son âne, et pense être bien sage.
Il n'est, dit le meunier, plus de veaux à mon âge :
Passez votre chemin, la fille, et m'en croyez.
Après maints quolibets coup sur coup renvoyés,
L'homme crut avoir tort, et mit son fils en croupe.
Au bout de trente pas, une troisième troupe
Trouve encore à gloser. L'un dit : Ces gens sont fous !
Le baudet n'en peut plus ; il mourra sous leurs coups.
Eh quoi ! charger ainsi cette pauvre bourrique !
N'ont-ils point de pitié de leur vieux domestique ?
Sans doute qu'à la foire ils vont vendre sa peau.
Parbleu ! dit le meunier, est bien fou du cerveau
Qui prétend contenter tout le monde et son père.
Essayons toutefois si par quelque manière
Nous en viendrons à bout. Ils descendent tous deux.
L'âne se prélassant marche seul devant eux.
Un quidam les rencontre et dit : Est-ce la mode
Que baudet aille à l'aise, et meunier s'incommode ?
Qui de l'âne ou du maître est fait pour se lasser ?
Je conseille à ces gens de le faire enchâsser.
Ils usent leurs souliers, et conservent leur âne !
Nicolas, au rebours : car, quand il va voir Jeanne,
Il monte sur sa bête ; et la chanson le dit.
Beau trio de baudets ! le meunier repartit :
Je suis âne, il est vrai, j'en conviens, je l'avoue ;
Mais que dorénavant on me blâme, on me loue,
Qu'on dise quelque chose ou qu'on ne dise rien,
J'en veux faire à ma tête. Il le fit, et fit bien.
Quant à vous, suivez Mars, ou l'Amour, ou le prince ;
Allez, venez, courez ; demeurez en province ;
Prenez femme, abbaye, emploi, gouvernement :
Les gens en parleront, n'en doutez nullement.


LE CHAT, LA BELETTE, ET LE PETIT LAPIN
Du palais d'un jeune lapin
Dame belette, un beau matin,
S'empara : c'est une rusée.
Le maître étant absent, ce lui fut chose aisée.
Elle porta chez lui ses pénates, un jour
Qu'il était allé faire à l'Aurore sa cour
Parmi le thym et la rosée.
Après qu'il eut brouté, trotté, fait tous ses tours,
Jeannot lapin retourne aux souterrains séjours.
La belette avait mis le nez à la fenêtre.
Ô dieux hospitaliers ! que vois-je ici paraître ?
Dit l'animal chassé du paternel logis.
Holà ! madame la belette,
Que l'on déloge sans trompette,
Ou je vais avertir tous les rats du pays.
La dame au nez pointu répondit que la terre
Était au premier occupant.
C'était un beau sujet de guerre,
Qu'un logis où lui-même il n'entrait qu'en rampant !
Et quand ce serait un royaume,
Je voudrais bien savoir, dit-elle, quelle loi
En a pour toujours fait l'octroi
À Jean, fils ou neveu de Pierre ou de Guillaume,
Plutôt qu'à Paul, plutôt qu'à moi.
Jean lapin allégua la coutume et l'usage :
Ce sont, dit-il, leurs lois qui m'ont de ce logis
Rendu maître et seigneur, et qui, de père en fils,
L'ont de Pierre à Simon, puis à moi Jean, transmis.
Le premier occupant, est-ce une loi plus sage ?
Or bien, sans crier davantage,
Rapportons-nous, dit-elle, à Raminagrobis.
C'était un chat vivant comme un dévot ermite,
Un chat faisant la chattemite,
Un saint homme de chat, bien fourré, gros et gras,
Arbitre expert sur tous les cas.
Jean lapin pour juge l'agrée.
Les voilà tous deux arrivés
Devant sa majesté fourrée.
Grippeminaud leur dit : Mes enfants, approchez,
Approchez : je suis sourd, les ans en sont la cause.
L'un et l'autre approcha, ne craignant nulle chose.
Aussitôt qu'à portée il vit les contestants,
Grippeminaud le bon apôtre,
Jetant des deux côtés la griffe en même temps,
Mit les plaideurs d'accord en croquant l'un et l'autre.
Ceci ressemble fort aux débats qu'ont parfois
Les petits souverains se rapportant aux rois.


LES DEUX MULETS
Deux mulets cheminaient, l'un d'avoine chargé,
L'autre portant l'argent de la gabelle.
Celui-ci, glorieux d'une charge si belle,
N'eût voulu pour beaucoup en être soulagé.
Il marchait d'un pas relevé
Et faisait sonner sa sonnette ;
Quand l'ennemi se présentant,
Comme il en voulait à l'argent,
Sur le mulet du fisc une troupe se jette,
Le saisit au frein, et l'arrête.
Le mulet, en se défendant,
Se sent percer de coups ; il gémit, il soupire.
Est-ce donc là, dit-il, ce qu'on m'avait promis ?
Ce mulet qui me suit du danger se retire,
Et moi j'y tombe et je péris !
Ami, lui dit son camarade,
Il n'est pas toujours bon d'avoir un haut emploi :
Si tu n'avais servi qu'un meunier comme moi,
Tu ne serais pas si malade.
"""

print(f"{len(corpus)} caractères")
print(corpus[:236])

38331 caractères
LA CIGALE ET LA FOURMI
La cigale, ayant chanté
Tout l'été,
Se trouva fort dépourvue
Quand la bise fut venue :
Pas un seul petit morceau
De mouche ou de vermisseau.
Elle alla crier famine
Chez la fourmi, sa voisine,
La priant de lui prêt


## 2. Le texte devient des nombres

Une machine ne calcule pas sur des lettres. On dresse donc la liste des
caractères distincts du corpus (son **vocabulaire**), et on donne à chacun un
numéro. Chaque numéro est un **token** (unité de texte). Ici, un token = un
caractère : c'est le découpage le plus simple qui existe. Les vrais LLMs
découpent en morceaux de mots ; tu construiras ce découpage au chapitre 7.

In [ ]:
chars = sorted(set(corpus))              # tous les caractères distincts, triés
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}   # caractère -> numéro
itos = {i: c for c, i in stoi.items()}       # numéro -> caractère


def encoder(texte):
    return [stoi[c] for c in texte]


def decoder(nombres):
    return "".join(itos[i] for i in nombres)


print(f"{vocab_size} caractères distincts : {''.join(chars[1:])!r}")
print(encoder("Le loup"))
print(decoder(encoder("Le loup")))

81 caractères distincts : ' !"\'(),-.:;?ABCDEFGHIJLMNOPQRSTUVXYabcdefghijlmnopqrstuvxyzÀÂÇÈÉÊÔÛàâçèéêîïôùûŒœ'
[23, 40, 1, 46, 49, 55, 50]
Le loup


In [ ]:
data = torch.tensor(encoder(corpus))     # tout le corpus, en une suite de numéros
print(data.shape, data.dtype)
print(corpus[:24], "->", data[:24].tolist())

torch.Size([38331]) torch.int64
LA CIGALE ET LA FOURMI
L -> [23, 13, 1, 15, 21, 19, 13, 23, 17, 1, 17, 31, 1, 23, 13, 1, 18, 26, 32, 29, 24, 21, 0, 23]


## 3. Le modèle : un fichier de nombres et une notice

La classe ci-dessous est la **notice** : elle dit quelles transformations
appliquer, dans quel ordre. Les nombres qu'elle utilise (les **weights**, les
poids) sont créés au hasard à l'instanciation ; c'est l'entraînement qui leur
donnera une valeur utile.

Le modèle lit les `block_size` derniers caractères et prédit un score pour
chaque caractère possible en sortie. Tu comprendras chaque couche au chapitre 6,
et les embeddings au chapitre 8.

In [ ]:
block_size = 16                          # le modèle lit 16 caractères pour prédire le 17e


class MiniLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.table = nn.Embedding(vocab_size, 24)          # chaque caractère -> 24 nombres
        self.reseau = nn.Sequential(
            nn.Linear(block_size * 24, 192),               # mélange tout le contexte
            nn.Tanh(),                                     # petite non-linéarité
            nn.Linear(192, vocab_size),                    # un score par caractère possible
        )

    def forward(self, x):
        e = self.table(x)                # (batch, block_size) -> (batch, block_size, 24)
        return self.reseau(e.flatten(1))  # aplati en (batch, block_size*24), puis scores


model = MiniLM().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params} nombres à apprendre")

91497 nombres à apprendre


In [ ]:
# Le forward en action : 4 contextes entrent, 4 lignes de scores sortent.
x_test = torch.randint(0, vocab_size, (4, block_size), device=device)
scores = model(x_test)
assert isinstance(scores, torch.Tensor), "le forward doit renvoyer un tenseur"
assert scores.shape == (4, vocab_size), (
    f"shape attendue (4, {vocab_size}), obtenue {tuple(scores.shape)}"
)
print("Forward OK : 4 contextes -> 4 lignes de", vocab_size, "scores")

Forward OK : 4 contextes -> 4 lignes de 81 scores


## 4. Écrire, caractère par caractère (échantillonnage)

Le modèle ne « rédige » pas. Il fait une seule chose : donner un score à chaque
caractère possible pour la suite. On transforme ces scores en probabilités, on
**tire au sort** le caractère suivant (le **sampling**, l'échantillonnage), on
l'ajoute au contexte, et on recommence. La `temperature` règle l'audace du
tirage : basse, on joue la sécurité ; haute, on prend des risques. Tout le
chapitre 5 est consacré à ce hasard contrôlé.

In [ ]:
def generer(prompt="\n", longueur=300, temperature=1.0):
    model.eval()
    ctx = ([stoi["\n"]] * block_size + encoder(prompt))[-block_size:]
    sortie = []
    with torch.no_grad():
        for _ in range(longueur):
            x = torch.tensor([ctx], device=device)
            probas = F.softmax(model(x) / temperature, dim=-1)     # scores -> probabilités
            i = torch.multinomial(probas, num_samples=1).item()    # tirage au sort
            sortie.append(itos[i])
            ctx = ctx[1:] + [i]          # on glisse d'un caractère et on recommence
    model.train()
    return prompt + "".join(sortie)

In [ ]:
# L'échantillonnage en action : bonne longueur, caractères du vocabulaire uniquement.
essai = generer("Le loup", longueur=50)
assert essai.startswith("Le loup"), "la sortie doit commencer par le prompt"
assert len(essai) == len("Le loup") + 50, "la sortie doit contenir `longueur` caractères de plus"
assert set(essai) <= set(chars), "tous les caractères générés doivent venir du vocabulaire"
print("Échantillonnage OK :", essai[:40].replace("\n", " ") + "…")

Échantillonnage OK : Le loupHBsoîlŒ?fdÇcÊ-x" tqïoâaPx'ÛsEsîmê…


## 5. Avant l'entraînement : la soupe

Les poids du modèle sont encore des nombres au hasard. Voyons ce qu'il « écrit »
dans cet état. Chaque caractère a environ une chance sur 81 de sortir : le
résultat est une soupe de caractères.

In [ ]:
print(generer("Le loup", longueur=300))

Le loupNâIdô; qMt"EÊF
?JtîEÈiOÉîRA?Hx)téùisuTSpSèPi.lù(J?qPî)XzMtTtprMÇ'"ùhRlYOâÂcD?êêYiMoçÔHàï(fSEJe.hôrhÉt-Ç)(vY'JIÔ(àeÂc;grn.uyyï'i!œ:pRge ebxtùù'vÇAÛêdry oetîG!:yÀ?Œgy'SgFVCn.T(À"ê.!)
ïuIÔÔRfhPûOtïT)VcNdPnG:âjeN,us-!SCêU"ùLœQÛTâdÊF d-çÂè?Âuu(MhRôùYàr!bTêÛUxsmYD!MÉ-Cyên.usyQM
UbtDÊâUPLûPÔiBaÈlb"vçfPÛutù


## 6. La boucle d'entraînement

Le refrain que tu retrouveras dans tout le livre : **prédire, mesurer l'erreur,
corriger les poids, recommencer**. À chaque étape, on tire 64 passages du corpus
au hasard, le modèle prédit le caractère qui suit chacun, la **loss** (la perte,
l'erreur) mesure à quel point il se trompe, et chaque poids est poussé d'un
cheveu dans la direction qui réduit cette erreur.

Comment trouve-t-on cette direction pour 90 000 nombres d'un coup ? C'est le
gradient : tout le chapitre 3, et tu construiras le moteur qui l'automatise au
chapitre 4.

In [ ]:
def fabriquer_batch(taille=64):
    ix = torch.randint(0, len(data) - block_size - 1, (taille,))       # 64 positions au hasard
    x = torch.stack([data[i : i + block_size] for i in ix]).to(device)  # 64 contextes
    y = data[ix + block_size].to(device)                                # le caractère qui suit chacun
    return x, y


x, y = fabriquer_batch()
print("contextes :", x.shape, "| cibles :", y.shape)
print(repr(decoder(x[0].tolist())), "->", repr(itos[y[0].item()]))

contextes : torch.Size([64, 16]) | cibles : torch.Size([64])
'notre conscience' -> '.'


In [ ]:
optimiseur = torch.optim.AdamW(model.parameters(), lr=1e-3)

debut = time.time()
for step in range(8001):
    x, y = fabriquer_batch()
    scores = model(x)                        # 1. prédire
    loss = F.cross_entropy(scores, y)        # 2. mesurer l'erreur
    optimiseur.zero_grad()                   # 3a. remettre les gradients à zéro
    loss.backward()                          # 3b. calculer les corrections
    optimiseur.step()                        # 3c. corriger les poids
    if step % 1000 == 0:
        print(f"étape {step:5d} | loss {loss.item():.2f}")
print(f"Entraînement terminé en {time.time() - debut:.0f} s")

étape     0 | loss 4.41
étape  1000 | loss 2.20
étape  2000 | loss 1.61
étape  3000 | loss 1.39
étape  4000 | loss 1.49
étape  5000 | loss 1.41
étape  6000 | loss 1.30
étape  7000 | loss 1.08
étape  8000 | loss 1.05
Entraînement terminé en 41 s


In [ ]:
# La preuve par la loss : bien plus basse qu'au départ (~4.4).
with torch.no_grad():
    pertes = []
    for _ in range(20):
        x, y = fabriquer_batch()
        pertes.append(F.cross_entropy(model(x), y).item())
    loss_moyenne = sum(pertes) / len(pertes)
print(f"loss moyenne après entraînement : {loss_moyenne:.2f}")
assert loss_moyenne < 2.0, (
    f"loss {loss_moyenne:.2f} trop haute : la boucle n'a pas appris (attendu < 2.0)"
)
print("Entraînement OK : le modèle a appris quelque chose.")

loss moyenne après entraînement : 0.91
Entraînement OK : le modèle a appris quelque chose.


## 7. Après l'entraînement

Mêmes poids, même notice, même fonction `generer`. Seule différence : les
90 000 nombres ont été corrigés 8 000 fois. Le texte n'est pas du vrai
français, mais ce n'est plus du bruit : des mots, des virgules, des retours à
la ligne de vers, un air de fable.

In [ ]:
print(generer("Le loup", longueur=400, temperature=0.8))

Le loup du voix derens quelle stis du son es leurs peillour cet souvant plun h'éprerre en entre anchiont à l'ausé biendit
Que couri boi veullermin ?
Ce cola dé le crans.


LEtCLETEURE ET LE POT DE LAIT
PEE MEt ET LAPvaux QÉainciqqu'à le vire ait.
C'ezmoi c pois, il cress ne sant celqu'au fetre Sant la contrement désinda fait,
Sans ec-, ends ce sévouteuxe :
Pours vollez, quents sagauss belastaplés.
Ne not


## 8. Joue avec la température

À toi : compare une température prudente (0.5), neutre (1.0) et audacieuse (1.3).
Puis change le prompt, la longueur, le nombre d'étapes d'entraînement. Casse des
choses. C'est ton laboratoire.

In [ ]:
for t in [0.3, 0.5, 1.0, 1.3, 2.0]:
    print(f"----- temperature = {t} -----")
    print(generer("La cigale", longueur=200, temperature=t))
    print()

----- temperature = 0.3 -----
La cigale renait dittis des couvait de coures,
L'ont de mit l'ancont la changant qu'à passient de pards, et de parde.
Quelque je la piant le bort, il pris de vel en chanteure.
Pous sare surait des démets, et c

----- temperature = 0.5 -----
La cigale de chures, la fit sant par quelle, ellerde le cons et tour chant alsée au plaux sont la porés ;
Mais on chatrei combe un je pass compes
Qu'il ne ca prau de manire, en content toutrous des les d'une v

----- temperature = 1.0 -----
La cigale devainnent aghonc lerres. fabde et élous : anchesi.
Pou-vons sens lun s pard éoutint,
Qu'an le mondetes de cospent du fouril.
La mèment en pudcer
Detes sont pons de cen.


LE Paplin vouvottille
Du de

----- temperature = 1.3 -----
La cigale relà parx,
À lanséamatéIabvezspans.
Eu le vit qu'il ne réjetice mainocraièrabler eufsétrit.
Elle- qui du c'est bacesos chansaè,
Que toupe éiS de viâde ? mansoé !
Pours : MeLsens patsant blarté batle 

----- temperature = 2.0 -----
La cig

## Exercices

À toi de jouer : trois exercices, du plus simple (●) au plus costaud (●●●).
Ce sont les trois gestes du chapitre, à refaire de tes mains : le forward du
modèle, l'échantillonnage, le refrain d'entraînement. Chaque cellule marquée
`# TODO(toi)` contient un trou ; complète-le, puis exécute la cellule de
validation (`assert`) qui suit : si elle passe sans erreur, c'est gagné.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta
place. Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est toi.
Les réponses sont dans le notebook solution
(`solutions/partie_1_etincelle/chapitre_01_faire_parler_une_machine_solution.ipynb`),
à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · Le forward, à la main — niveau ●

Le trajet d'un contexte dans le modèle : la table d'embedding transforme les
16 numéros en 16 paquets de 24 nombres, on aplatit, et le réseau rend un score
par caractère possible. Réécris ce trajet toi-même, en réutilisant
`model.table` et `model.reseau` : ta fonction doit rendre exactement les mêmes
scores que `model(x)`.

In [ ]:
def forward_manuel(x):
    e = model.table(x)               # (batch, block_size) -> (batch, block_size, 24)
    e_flattened = e.flatten(1)         # TODO(toi) : aplatis e pour obtenir une shape (batch, block_size * 24),
    scores = model.reseau(e_flattened) # puis passe le résultat dans model.reseau et renvoie les scores.
    # Indice : e.flatten(1) aplatit toutes les dimensions sauf la première.
    return scores

In [ ]:
# Validation : le forward, à la main.
x_test = torch.randint(0, vocab_size, (4, block_size), device=device)
scores = forward_manuel(x_test)
assert isinstance(scores, torch.Tensor), "forward_manuel doit renvoyer un tenseur"
assert scores.shape == (4, vocab_size), (
    f"shape attendue (4, {vocab_size}), obtenue {tuple(scores.shape)}"
)
assert torch.allclose(scores, model(x_test)), (
    "les scores doivent être identiques à ceux de model(x_test)"
)
print("Forward OK : 4 contextes -> 4 lignes de", vocab_size, "scores, identiques au modèle")

Forward OK : 4 contextes -> 4 lignes de 81 scores, identiques au modèle


### Exercice 2 · Échantillonner toi-même — niveau ●●

Le rituel de l'écriture : des scores, des probabilités, un tirage au sort, et
on recommence. Réécris le cœur de `generer` dans ta propre fonction. Les trois
gestes sont détaillés en commentaire ; la température s'applique aux scores,
AVANT le softmax.

In [ ]:
def ma_generation(prompt="\n", longueur=300, temperature=1.0):
    model.eval()
    ctx = ([stoi["\n"]] * block_size + encoder(prompt))[-block_size:]
    sortie = []
    with torch.no_grad():
        for _ in range(longueur):
            x = torch.tensor([ctx], device=device)
            # TODO(toi) : trois gestes.
            logits = model(x) / temperature  # 1) calcule les scores : model(x), puis divise-les par temperature ;
            probas = F.softmax(logits, dim=-1)  # 2) transforme-les en probabilités avec F.softmax(..., dim=-1) ;
            # 3) tire un numéro au sort avec torch.multinomial(probas, num_samples=1).item()
            i = torch.multinomial(probas, num_samples=1).item()
            sortie.append(itos[i])
            ctx = ctx[1:] + [i]          # on glisse d'un caractère et on recommence
    model.train()
    return prompt + "".join(sortie)

In [ ]:
# Validation : échantillonnage. Bonne longueur, caractères du vocabulaire uniquement.
essai = ma_generation("Le loup", longueur=50)
assert essai.startswith("Le loup"), "la sortie doit commencer par le prompt"
assert len(essai) == len("Le loup") + 50, "la sortie doit contenir `longueur` caractères de plus"
assert set(essai) <= set(chars), "tous les caractères générés doivent venir du vocabulaire"
print("Échantillonnage OK :", essai[:40].replace("\n", " ") + "…")

Échantillonnage OK : Le loup ronive topage ! riente, Vous me …


### Exercice 3 · Le refrain d'entraînement — niveau ●●●

Le geste le plus important du livre : prédire, mesurer l'erreur, corriger les
poids, recommencer. Repars d'un modèle tout neuf (`model2`, poids au hasard) et
entraîne-le toi-même. Si ta boucle est juste, sa loss fondra d'environ 4.4 vers
moins de 1.

In [ ]:
model2 = MiniLM().to(device)             # un modèle tout neuf, poids au hasard
optimiseur2 = torch.optim.AdamW(model2.parameters(), lr=1e-3)

for step in range(8001):
    x, y = fabriquer_batch()
    # TODO(toi) : le refrain, en quatre gestes, sur model2 et optimiseur2.
    scores = model2(x)# 1) prédire : scores = model2(x)
    loss = F.cross_entropy(scores, y)  # 2) mesurer l'erreur : loss = F.cross_entropy(scores, y)
    optimiseur2.zero_grad()  # 3) remettre les gradients à zéro : optimiseur2.zero_grad()
    loss.backward()#    puis calculer les corrections : loss.backward()
    optimiseur2.step() # 4) corriger les poids : optimiseur2.step()
    # loss = F.cross_entropy(scores, y)
    if step % 1000 == 0:
        print(f"étape {step:5d} | loss {loss.item():.2f}")

étape     0 | loss 4.44
étape  1000 | loss 1.95
étape  2000 | loss 1.52
étape  3000 | loss 1.73
étape  4000 | loss 1.39
étape  5000 | loss 1.31
étape  6000 | loss 1.00
étape  7000 | loss 0.79
étape  8000 | loss 0.80


In [ ]:
# Validation : entraînement. La loss doit être bien plus basse qu'au départ (~4.4).
with torch.no_grad():
    pertes = []
    for _ in range(20):
        x, y = fabriquer_batch()
        pertes.append(F.cross_entropy(model2(x), y).item())
    loss_moyenne = sum(pertes) / len(pertes)
print(f"loss moyenne de model2 après entraînement : {loss_moyenne:.2f}")
assert loss_moyenne < 2.0, (
    f"loss {loss_moyenne:.2f} trop haute : la boucle n'a pas appris (attendu < 2.0)"
)
print("Entraînement OK : ton refrain fonctionne.")

loss moyenne de model2 après entraînement : 0.84
Entraînement OK : ton refrain fonctionne.


## Et maintenant ?

Trois validations vertes ? Tu viens d'entraîner un modèle de langage, en entier :
texte devenu nombres, notice, poids appris, boucle d'entraînement,
échantillonnage. Chaque bloc de ce notebook devient un chapitre du livre :

- le découpage en tokens : chapitre 7 ;
- la mesure d'erreur et le gradient : chapitres 3 et 4 ;
- le réseau (couches, embeddings) : chapitres 6 et 8 ;
- l'échantillonnage et la température : chapitre 5 ;
- ce qui manque encore à ce petit modèle (l'attention, le Transformer) :
  chapitres 9 et 10.

Retour au livre pour la suite : *Chapitre 2 · Les nombres qui apprennent*.